# Fine-Tuning LLaMA-3.2-3B for Relation Extraction
This notebook demonstrates how to fine-tune a LLaMA-3.2-3B model to classify 'at' and 'isAt' relations using the Unsloth library for efficient training.

## 1. Setup and Installation
First, we install Unsloth and other required dependencies.

In [ ]:
# Official Unsloth Nuclear installation for Colab
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes wandb

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-7mv9mbei/unsloth_dfdb28dc12b544c598595f09aa5d6759
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-7mv9mbei/unsloth_dfdb28dc12b544c598595f09aa5d6759
  Resolved https://github.com/unslothai/unsloth.git to commit c8bcacc3fea27e97bea0ea09c9ad7554c729724f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 139.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 20.9 MB/s eta 0:00:00
   

## 2. Environment Setup
Mount Google Drive to access the datasets and save our trained models, then import the necessary libraries.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import gc
import torch
import wandb
import json
from datasets import Dataset
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForSeq2Seq

Mounted at /content/drive
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


## 3. Data Preparation
Define the prompt structures for the classification tasks and create a helper function to load and format the JSON datasets into Hugging Face `Dataset` objects.

In [ ]:
at_definitions = """• at=TRUE: Explicit evidence of residency, origin, or long-term role.
• at=PROBABLE: Implicit cues or professional affiliation make a relation a likely assumption.
• at=FALSE: No evidence or contradictory evidence."""

isAt_definitions = """• isAt=TRUE: Explicit evidence the person was at the location within one month of the publication date.
• isAt=FALSE: Event occurred in the past or person is elsewhere."""

base_prompt = """TASK: Classify {relation} between Person: {person} and Place: {place}.
DEFINITIONS: ### DEFINITIONS:
{definitions}
TEXT: {text}"""

def load_and_format_dataset(json_path, relation_type):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    definitions = at_definitions if relation_type == "at" else isAt_definitions

    formatted_data = []
    for item in data:
        user_msg = base_prompt.format(
            relation=relation_type,
            definitions=definitions,
            person=item.get('person', 'Unknown'),
            place=item.get('place', 'Unknown'),
            text=item.get('text', '')
        )
        messages = [
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": str(item.get('label', ''))}
        ]
        formatted_data.append({"messages": messages})
    return Dataset.from_list(formatted_data)

## 4. Core Training Setup
Define a custom callback for early stopping based on target loss, and construct the main `train_adapter` function which handles model initialization with LoRA, data tokenization, and the `SFTTrainer` setup.

In [ ]:
from transformers import TrainerCallback

# Custom Callback to stop training when training loss hits target
class StopOnLossCallback(TrainerCallback):
    def __init__(self, target_loss=0.5):
        self.target_loss = target_loss

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            logs = {}
        # Check the training loss logged by the Trainer
        current_loss = logs.get("loss")
        if current_loss is not None and current_loss <= self.target_loss:
            print(f"\nTraining loss reached {current_loss} (<= {self.target_loss}). Stopping training.")
            control.should_training_stop = True

# Common Model & Training Setup Function
def train_adapter(relation_type, train_dataset_path, eval_dataset_path, save_dir):
    max_seq_length = 4096
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
        max_seq_length=max_seq_length,
        dtype=None,
        load_in_4bit=True,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_alpha=64,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
        use_rslora=False,
    )

    tokenizer = get_chat_template(
        tokenizer,
        chat_template="llama-3.1",
    )

    def apply_template(examples):
        texts = [tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
                 for messages in examples["messages"]]
        return {"text": texts}

    train_dataset = load_and_format_dataset(train_dataset_path, relation_type)
    train_dataset = train_dataset.map(apply_template, batched=True)

    eval_dataset = load_and_format_dataset(eval_dataset_path, relation_type)
    eval_dataset = eval_dataset.map(apply_template, batched=True)

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
        dataset_num_proc=2,
        packing=False,
        callbacks=[StopOnLossCallback(target_loss=0.5)], # Add our custom callback
        args = SFTConfig(
          output_dir = save_dir,         # Use the unique Drive path
          report_to = "wandb",
          run_name = f"llama-3b-{relation_type}-no-reasoning-early-stopping",

          num_train_epochs = 5,          # Increased to 5 for better logic capture
          max_steps = -1,

          # Checkpointing & Evaluation
          eval_strategy = "steps",
          eval_steps = 100,
          save_strategy = "steps",
          save_steps = 100,
          load_best_model_at_end = True,
          metric_for_best_model = "loss",
          save_total_limit = 1,

          learning_rate = 1e-4,
          per_device_train_batch_size = 2,
          gradient_accumulation_steps = 4,
          lr_scheduler_type = "cosine",
          weight_decay = 0.05,           # Higher decay to force logic over memory

          # Hardware
          bf16 = True,
          max_seq_length = 4096,
          dataset_text_field = "text",
          packing = False,
          seed = 3407,
      )
    )

    trainer.train()

    # Save adapter
    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)
    print(f"Saved {relation_type} adapter to {save_dir}")

    return model, trainer


## 5. Training the 'at' Relation Adapter
Execute the training pipeline for the 'at' dataset.

In [ ]:
# 1. Train 'at' adapter
at_train_path = '/content/drive/MyDrive/colab_data/HIPE-2026-data/data/sandbox/at_train.json'
at_eval_path = '/content/drive/MyDrive/colab_data/HIPE-2026-data/data/sandbox/at_eval.json'
at_save_dir = '/content/drive/MyDrive/colab_data/HIPE-2026-data/trained_models/llama_3b_at_adapter_no_reasoning_early_stopping'

# NOTE: You will need to log into wandb when this runs if you haven't already
model_at, trainer_at = train_adapter("at", at_train_path, at_eval_path, at_save_dir)

==((====))==  Unsloth 2026.6.7: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

...

wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
50,2.262412,2.379838
100,2.266146,2.235368
150,2.034209,2.059572
200,1.725894,1.888129
250,1.458012,1.707130
300,1.638273,1.540927
350,1.353907,1.372849
400,0.921515,1.227970
450,1.095249,1.064176
500,0.630844,0.920004


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

Saved at adapter to /content/drive/MyDrive/trained_models/llama_3b_at_adapter


In [ ]:
# 2. Clear VRAM & Close wandb run
import wandb
wandb.finish()

del model_at
del trainer_at
gc.collect()
torch.cuda.empty_cache()
print("VRAM Cleared and wandb run closed.")

VRAM Cleared.


## 6. Training the 'isAt' Relation Adapter
Execute the training pipeline for the 'isAt' dataset.

In [ ]:
# 3. Train 'isAt' adapter
isAt_train_path = '/content/drive/MyDrive/colab_data/HIPE-2026-data/data/sandbox/isAt_train.json'
isAt_eval_path = '/content/drive/MyDrive/colab_data/HIPE-2026-data/data/sandbox/isAt_eval.json'
isAt_save_dir = '/content/drive/MyDrive/colab_data/HIPE-2026-data/trained_models/llama_3b_isAt_adapter_no_reasoning_early_stopping'

model_isAt, trainer_isAt = train_adapter("isAt", isAt_train_path, isAt_eval_path, isAt_save_dir)

==((====))==  Unsloth 2026.6.7: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.


Map:   0%|          | 0/1084 [00:00<?, ? examples/s]

Map:   0%|          | 0/270 [00:00<?, ? examples/s]

...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,084 | Num Epochs = 3 | Total steps = 408
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 48,627,712 of 3,261,377,536 (1.49% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
50,2.249755,2.304170
100,2.120750,2.133632
150,1.898294,1.981956
200,1.488945,1.849738
250,1.718789,1.722963
300,0.791176,1.636091
350,1.676393,1.591750
400,1.110291,1.581104
408,0.566984,1.580867


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

Saved isAt adapter to /content/drive/MyDrive/trained_models/llama_3b_isAt_adapter


## 7. Continuation Training (Optional)
In case we need to train the adapters for more epochs, this function loads an existing adapter and resumes the training process.

In [ ]:
from peft import PeftModel

def continue_training_adapter(relation_type, train_dataset_path, eval_dataset_path, adapter_path, new_save_dir):
    max_seq_length = 4096

    # 1. Load the base model
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
        max_seq_length=max_seq_length,
        dtype=None,
        load_in_4bit=True,
    )

    # 2. Load the pre-saved adapter and make sure it is trainable
    model = PeftModel.from_pretrained(model, adapter_path, is_trainable=True)

    tokenizer = get_chat_template(
        tokenizer,
        chat_template="llama-3.1",
    )

    def apply_template(examples):
        texts = [tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
                 for messages in examples["messages"]]
        return {"text": texts}

    train_dataset = load_and_format_dataset(train_dataset_path, relation_type)
    train_dataset = train_dataset.map(apply_template, batched=True)

    eval_dataset = load_and_format_dataset(eval_dataset_path, relation_type)
    eval_dataset = eval_dataset.map(apply_template, batched=True)

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
        dataset_num_proc=2,
        packing=False,
        callbacks=[StopOnLossCallback(target_loss=0.5)], # Add our custom callback
        args = SFTConfig(
          output_dir = new_save_dir,
          report_to = "wandb",
          run_name = f"llama-3b-{relation_type}-specialist",

          num_train_epochs = 3,          # 3 more epochs
          max_steps = -1,

          eval_strategy = "steps",
          eval_steps = 100,
          save_strategy = "steps",
          save_steps = 100,
          load_best_model_at_end = True,
          metric_for_best_model = "loss",
          save_total_limit = 1,

          learning_rate = 5e-5,          # Lowered for continuation training
          per_device_train_batch_size = 2,
          gradient_accumulation_steps = 4,
          lr_scheduler_type = "cosine",
          weight_decay = 0.05,

          bf16 = True,
          max_seq_length = 4096,
          dataset_text_field = "text",
          packing = False,
          seed = 3407,
      )
    )

    trainer.train()

    # Save the newly trained adapter
    model.save_pretrained(new_save_dir)
    tokenizer.save_pretrained(new_save_dir)
    print(f"Saved continued {relation_type} adapter to {new_save_dir}")

    return model, trainer


### Run the continuation training
This uses the function above to load the previous adapters and train them for 3 more epochs. Note that it saves them in a new folder appending `_v2` so we do not overwrite the original weights.

In [ ]:
# Paths for 'at' relation continuation
at_train_path = '/content/drive/MyDrive/colab_data/HIPE-2026-data/data/sandbox/at_train.json'
at_eval_path = '/content/drive/MyDrive/colab_data/HIPE-2026-data/data/sandbox/at_eval.json'
old_at_adapter_path = '/content/drive/MyDrive/colab_data/HIPE-2026-data/trained_models/llama_3b_at_adapter'
new_at_save_dir = '/content/drive/MyDrive/colab_data/HIPE-2026-data/trained_models/llama_3b_at_adapter_v2'

# Run continuation for 'at'
model_at_cont, trainer_at_cont = continue_training_adapter(
    "at",
    at_train_path,
    at_eval_path,
    old_at_adapter_path,
    new_at_save_dir
)

# Clear VRAM after training 'at'
wandb.finish()
del model_at_cont
del trainer_at_cont
gc.collect()
torch.cuda.empty_cache()

# Paths for 'isAt' relation continuation
isAt_train_path = '/content/drive/MyDrive/colab_data/HIPE-2026-data/data/sandbox/isAt_train.json'
isAt_eval_path = '/content/drive/MyDrive/colab_data/HIPE-2026-data/data/sandbox/isAt_eval.json'
old_isAt_adapter_path = '/content/drive/MyDrive/colab_data/HIPE-2026-data/trained_models/llama_3b_isAt_adapter'
new_isAt_save_dir = '/content/drive/MyDrive/colab_data/HIPE-2026-data/trained_models/llama_3b_isAt_adapter_v2'

# Run continuation for 'isAt'
model_isAt_cont, trainer_isAt_cont = continue_training_adapter(
    "isAt",
    isAt_train_path,
    isAt_eval_path,
    old_isAt_adapter_path,
    new_isAt_save_dir
)

# Clear VRAM after training 'isAt'
wandb.finish()
del model_isAt_cont
del trainer_isAt_cont
gc.collect()
torch.cuda.empty_cache()

==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

...

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/270 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,084 | Num Epochs = 3 | Total steps = 408
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 48,627,712 of 3,261,377,536 (1.49% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

### Tomorrow's Run: Train ONLY the `isAt` adapter
Run this cell tomorrow to complete the training for the `isAt` adapter without re-running the `at` adapter.

In [ ]:
# Paths for 'isAt' relation continuation
isAt_train_path = '/content/drive/MyDrive/colab_data/HIPE-2026-data/data/sandbox/isAt_train.json'
isAt_eval_path = '/content/drive/MyDrive/colab_data/HIPE-2026-data/data/sandbox/isAt_eval.json'
old_isAt_adapter_path = '/content/drive/MyDrive/colab_data/HIPE-2026-data/trained_models/llama_3b_isAt_adapter'
new_isAt_save_dir = '/content/drive/MyDrive/colab_data/HIPE-2026-data/trained_models/llama_3b_isAt_adapter_v2'

# Run continuation for 'isAt' only
print("Starting isAt continuation training...")
model_isAt_cont, trainer_isAt_cont = continue_training_adapter(
    "isAt",
    isAt_train_path,
    isAt_eval_path,
    old_isAt_adapter_path,
    new_isAt_save_dir
)

# Clear VRAM after training 'isAt'
import wandb
import gc
import torch

wandb.finish()
del model_isAt_cont
del trainer_isAt_cont
gc.collect()
torch.cuda.empty_cache()
print("Finished training isAt and cleared VRAM!")

Starting isAt continuation training...
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

...

Step,Training Loss,Validation Loss
100,0.530581,1.444213
200,0.434520,1.292962
300,0.309526,1.189190
400,0.629719,1.162947
408,0.140600,1.162883


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

Saved continued isAt adapter to /content/drive/MyDrive/colab_data/HIPE-2026-data/trained_models/llama_3b_isAt_adapter_v2


eval/loss,█▄▂▁▁
eval/runtime,▇▇▁█▃
eval/samples_per_second,▁▁█▁▄
eval/steps_per_second,▁▁▁▁▁
train/epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▆▆▆▇▇▇▇▇▇███
train/global_step,▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▄▄▄▄▄▄▄▅▅▆▆▆▆▇▇▇▇▇▇▇█████
train/grad_norm,▃▃▃▃▂▂▃█▃▄▃▅▃▃▃▃▃▃▅▄▃▂▁▅▂▃▃▃▄▃▃▃▂▃▂▂▃▂▁▁
train/learning_rate,▂▄▅███████▇▇▇▇▇▆▆▅▅▅▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
train/loss,█▇▄▇▅▇▃▆█▆█▅▅▁▅▅▇▅▆▆▄▆▅▄▁▃▇▃▃▂▂▄▄▄▁▆▃▅▃▂
eval/loss,1.16288
eval/runtime,138.7274


Finished training isAt and cleared VRAM!
